# Modelo CRNN: CNN 1D + BiLSTM para Clasificación de Comandos de Voz

In [ ]:
# Colab setup
try:
    import google.colab
    IN_COLAB = True
    !pip install -q kagglehub torch torchaudio pandas numpy matplotlib scikit-learn
    import kagglehub
    DATA_PATH = kagglehub.competition_download("voice-commands-classification-2026")
    import os
    DATA_PATH = os.path.join(DATA_PATH, 'train')
except ImportError:
    IN_COLAB = False
    DATA_PATH = './data/train'

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

In [ ]:
SAMPLE_RATE = 16000
BATCH_SIZE = 64
EPOCHS = 20
LR = 0.001
N_CLASSES = 35
N_CHANNEL = 32
PATIENCE = 5
VAL_SIZE = 0.2
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
class VoiceCommandsDataset(Dataset):
    def __init__(self, df, audio_dir, max_samples=SAMPLE_RATE):
        self.df = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.max_samples = max_samples

    def __len__(self):
        return len(self.df)

    def _normalize_length(self, audio):
        if audio.shape[0] > self.max_samples:
            return audio[:self.max_samples]
        if audio.shape[0] < self.max_samples:
            padding = self.max_samples - audio.shape[0]
            return np.pad(audio, (0, padding), mode='constant')
        return audio

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = self.audio_dir / row['filename']
        arr = np.load(file_path, allow_pickle=False).reshape(-1).astype(np.float32)
        audio = self._normalize_length(arr)
        label = int(row['label_id'])
        return torch.tensor(audio, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [ ]:
AUDIO_DIR = os.path.join(DATA_PATH, 'train', 'train')

train_df = pd.read_csv('train_metadata.csv')
train_df = train_df.dropna(subset=['label']).copy()
train_df['label'] = train_df['label'].astype(str)

label_encoder = LabelEncoder()
train_df['label_id'] = label_encoder.fit_transform(train_df['label'])
num_classes = len(label_encoder.classes_)
print(f"Numero de clases: {num_classes}")
print("Clases:", list(label_encoder.classes_))

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df['label_id']
)

train_dataset = VoiceCommandsDataset(train_split_df, audio_dir=AUDIO_DIR)
val_dataset = VoiceCommandsDataset(val_split_df, audio_dir=AUDIO_DIR)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"Total muestras: {len(train_df)}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
class CRNNModel(nn.Module):
    def __init__(self, n_input=1, n_output=35, n_channel=32):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(n_input, n_channel, kernel_size=80, stride=16),
            nn.BatchNorm1d(n_channel),
            nn.ReLU()
        )
        self.rnn = nn.LSTM(
            input_size=n_channel,
            hidden_size=n_channel,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(2 * n_channel, n_output)

    def forward(self, x):
        # x: (batch, n_input=1, time)
        x = self.cnn(x)  # (batch, n_channel, T')
        x = x.permute(0, 2, 1)  # (batch, T', n_channel)
        x, _ = self.rnn(x)  # (batch, T', 2*n_channel)
        x = x.mean(dim=1)  # (batch, 2*n_channel)
        x = self.classifier(x)  # (batch, n_output)
        return x

In [ ]:
model = CRNNModel(n_input=1, n_output=num_classes, n_channel=N_CHANNEL).to(device)
print(model)

# Verificar con un batch dummy
dummy = torch.randn(2, 1, SAMPLE_RATE).to(device)
out = model(dummy)
print(f"Salida shape: {out.shape}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

In [ ]:
best_val_loss = float('inf')
patience_counter = 0
train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for waveforms, labels in train_loader:
        waveforms = waveforms.unsqueeze(1).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(waveforms)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * waveforms.size(0)
        preds = outputs.argmax(dim=1)
        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

    epoch_train_loss = train_loss_sum / train_total
    epoch_train_acc = train_correct / train_total
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for waveforms, labels in val_loader:
            waveforms = waveforms.unsqueeze(1).to(device)
            labels = labels.to(device)
            outputs = model(waveforms)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * waveforms.size(0)
            preds = outputs.argmax(dim=1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    epoch_val_loss = val_loss_sum / val_total
    epoch_val_acc = val_correct / val_total
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc)

    scheduler.step(epoch_val_loss)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc * 100:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc * 100:.2f}%"
    )

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_crnn.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping en epoca {epoch + 1}")
            break

print("Entrenamiento completado.")

In [ ]:
model.load_state_dict(torch.load('best_crnn.pth', map_location=device))
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for waveforms, labels in val_loader:
        waveforms = waveforms.unsqueeze(1).to(device)
        labels = labels.to(device)
        logits = model(waveforms)
        preds = logits.argmax(dim=1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(labels.cpu().numpy().tolist())

val_acc = (np.array(y_true) == np.array(y_pred)).mean()
print(f"Accuracy en validacion: {val_acc * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_title('Perdida por epoca')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Cross Entropy')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(train_accs, label='Train Acc')
axes[1].plot(val_accs, label='Val Acc')
axes[1].set_title('Accuracy por epoca')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(np.float32) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(10, 8))
plt.imshow(cm_norm, interpolation='nearest', aspect='auto')
plt.title('Matriz de confusion (normalizada)')
plt.colorbar()

tick_marks = np.arange(len(label_encoder.classes_))
plt.xticks(tick_marks, label_encoder.classes_, rotation=90)
plt.yticks(tick_marks, label_encoder.classes_)
plt.xlabel('Prediccion')
plt.ylabel('Etiqueta real')
plt.tight_layout()
plt.show()

print('Reporte de clasificacion (validacion):')
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, zero_division=0))